# Ultralytics Models on DIVA-HisDB (YOLO / RT-DETR / SAM)

Lets you choose one DIVA-HisDB subset and one Ultralytics model family in Colab, copy the data from Drive to local SSD, and then train or preview the selected model.

- YOLO: segmentation training, random-test preview, and HSCP evaluation.
- RT-DETR: detection training and random-test preview.
- SAM: pretrained random-test preview only.

Assumes the prepared `DIVA-HisDB/` folder is already on Drive under `fsl-tl-manuscripts/00_data/`.

**Runtime:** GPU (T4 / L4 / A100). Pick via `Runtime > Change runtime type`.

## 1  Pick dataset + model

In [ ]:
DATASET = 'cs863'  # 'cs18' | 'cs863' | 'cb55'
MODEL_FAMILY = 'yolo'  # 'yolo' | 'rtdetr' | 'sam'

DEFAULT_MODEL_WEIGHTS = {
    'yolo': 'yolov8n-seg.pt',
    'rtdetr': 'rtdetr-l.pt',
    'sam': 'mobile_sam.pt',
}
MODEL_WEIGHTS = DEFAULT_MODEL_WEIGHTS[MODEL_FAMILY]

FORCE_RETRAIN = False
FORCE_REBUILD_DATASET = False
FORCE_REEXPORT_HSCP = True

EPOCHS = 100
IMGSZ = 1024
BATCH = 4
WORKERS = 2
PATIENCE = 20
PRED_CONF = 0.5
HSCP_IOU_THRESH = 0.75

MODEL_KIND = MODEL_FAMILY.lower()
if MODEL_KIND not in DEFAULT_MODEL_WEIGHTS:
    raise ValueError(f'Unsupported model family: {MODEL_FAMILY}')

TRAINABLE_MODEL_FAMILIES = {'yolo', 'rtdetr'}
HSCP_MODEL_FAMILIES = {'yolo'}
RUN_SUFFIX = {'yolo': 'seg', 'rtdetr': 'det', 'sam': 'sam'}[MODEL_KIND]
OUTPUT_GROUP = {'yolo': 'segmentation', 'rtdetr': 'detection', 'sam': 'foundation'}[MODEL_KIND]

MODEL_STEM = MODEL_WEIGHTS.replace('-seg.pt', '').replace('.pt', '')
RUN_NAME = f'ultralytics_{DATASET}_{MODEL_STEM}_{RUN_SUFFIX}'

DIVA_DRIVE = '/content/drive/MyDrive/fsl-tl-manuscripts/00_data/DIVA-HisDB'
OUTPUT_DRIVE = f'/content/drive/MyDrive/fsl-tl-manuscripts/80_models/{OUTPUT_GROUP}/{RUN_NAME}'
OUTPUT_LOCAL = f'/content/output/{RUN_NAME}'
EVAL_DRIVE = f'/content/drive/MyDrive/fsl-tl-manuscripts/99_evaluation/{RUN_NAME}'
EVAL_LOCAL = f'/content/output/eval_{RUN_NAME}'

## 2  Mount Drive + mirror to local SSD

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil
from pathlib import Path

DIVA_LOCAL = Path('/content/DIVA-HisDB')
RUN_LOCAL = Path(OUTPUT_LOCAL)
RUN_DRIVE = Path(OUTPUT_DRIVE)
EVAL_RUN_LOCAL = Path(EVAL_LOCAL)
EVAL_RUN_DRIVE = Path(EVAL_DRIVE)

if not DIVA_LOCAL.exists():
    print('copying DIVA-HisDB to /content ...')
    shutil.copytree(DIVA_DRIVE, DIVA_LOCAL)

if MODEL_KIND in TRAINABLE_MODEL_FAMILIES and RUN_DRIVE.exists() and not RUN_LOCAL.exists() and not FORCE_RETRAIN:
    print('restoring previous run to /content ...')
    shutil.copytree(RUN_DRIVE, RUN_LOCAL)

print('ready:', DIVA_LOCAL, RUN_LOCAL)

## 3  Install deps

In [ ]:
%pip install -q ultralytics pandas matplotlib pillow pyyaml tqdm opencv-python-headless

In [ ]:
import torch
import ultralytics

print('torch      :', torch.__version__, '| cuda', torch.version.cuda)
print('ultralytics:', ultralytics.__version__)
if torch.cuda.is_available():
    print('GPU        :', torch.cuda.get_device_name(0))
else:
    print('GPU        : not available')

## 4  Build dataset (YOLO and RT-DETR only)

In [ ]:
from __future__ import annotations

import shutil
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path
from uuid import uuid4
from xml.dom import minidom

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
import torch
import yaml
from ultralytics import RTDETR, SAM, YOLO
from ultralytics.data.converter import convert_coco

DIVA_INFO = {
    'CB55': {
        'coco_dir': 'coco_dataset_CB55',
        'image_dir': 'CB55/img-CB55/img',
        'task2_page_dir': 'CB55/PAGE-gt-CB55-TASK-2/TASK-2',
    },
    'CS18': {
        'coco_dir': 'coco_dataset_CS18',
        'image_dir': 'CS18/img-CS18/img',
        'task2_page_dir': 'CS18/PAGE-gt-CS18-TASK-2/TASK-2',
    },
    'CS863': {
        'coco_dir': 'coco_dataset_CS863',
        'image_dir': 'CS863/img-CS863/img',
        'task2_page_dir': 'CS863/PAGE-gt-CS863-TASK-2/TASK-2',
    },
}
IMAGE_SUFFIXES = ('.jpg', '.png', '.tif', '.tiff')
PAGE_NS = 'http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15'

subset_key = DATASET.upper()
if subset_key not in DIVA_INFO:
    raise ValueError(f'Unsupported dataset: {DATASET}')

subset_info = DIVA_INFO[subset_key]
diva_root = Path(DIVA_LOCAL)
COCO_ROOT = diva_root / subset_info['coco_dir']
SOURCE_SPLITS = {
    'train': diva_root / subset_info['image_dir'] / 'training',
    'val': diva_root / subset_info['image_dir'] / 'validation',
    'test': diva_root / subset_info['image_dir'] / 'public-test',
}
TEST_PAGE_DIR = diva_root / subset_info['task2_page_dir'] / 'public-test'
SEG_DATASET_ROOT = diva_root / f'yolo_seg_dataset_{subset_key}'
DET_DATASET_ROOT = diva_root / f'yolo_dataset_{subset_key}'
ACTIVE_DATASET_ROOT = None
if MODEL_KIND == 'yolo':
    ACTIVE_DATASET_ROOT = SEG_DATASET_ROOT
elif MODEL_KIND == 'rtdetr':
    ACTIVE_DATASET_ROOT = DET_DATASET_ROOT
DATASET_YAML = None if ACTIVE_DATASET_ROOT is None else ACTIVE_DATASET_ROOT / 'dataset.yaml'
RUN_DIR = Path(OUTPUT_LOCAL)
BEST_WEIGHTS = RUN_DIR / 'weights' / 'best.pt'
PRED_XML_DIR = Path(EVAL_LOCAL) / 'page_xml'
HSCP_CSV_PATH = Path(EVAL_LOCAL) / 'hscp_results.csv'
DEVICE = 0 if torch.cuda.is_available() else 'cpu'


def iter_split_images(split_name: str) -> list[Path]:
    image_paths = []
    for suffix in IMAGE_SUFFIXES:
        image_paths.extend(sorted(SOURCE_SPLITS[split_name].glob(f'*{suffix}')))
    return sorted(image_paths)


def copy_split_images(dataset_root: Path) -> None:
    for split_name in SOURCE_SPLITS:
        target_dir = dataset_root / 'images' / split_name
        target_dir.mkdir(parents=True, exist_ok=True)
        for image_path in iter_split_images(split_name):
            target_path = target_dir / image_path.name
            if not target_path.exists():
                shutil.copy2(image_path, target_path)


def convert_coco_annotations(dataset_root: Path, use_segments: bool) -> None:
    expected_labels = [dataset_root / 'labels' / split_name for split_name in SOURCE_SPLITS]
    if all(path.exists() and any(path.glob('*.txt')) for path in expected_labels):
        return

    temp_root = dataset_root.parent / f'{subset_key.lower()}_{MODEL_KIND}_{uuid4().hex}'
    try:
        convert_coco(
            labels_dir=str(COCO_ROOT),
            save_dir=str(temp_root),
            use_segments=use_segments,
            cls91to80=False,
        )
        labels_dir = temp_root / 'labels'
        target_labels = dataset_root / 'labels'
        target_labels.mkdir(parents=True, exist_ok=True)
        for split_name in SOURCE_SPLITS:
            split_source = labels_dir / split_name
            split_target = target_labels / split_name
            if split_target.exists():
                shutil.rmtree(split_target)
            shutil.move(str(split_source), str(split_target))
    finally:
        shutil.rmtree(temp_root, ignore_errors=True)


def write_dataset_yaml(dataset_root: Path) -> None:
    config = {
        'path': str(dataset_root),
        'train': 'images/train',
        'val': 'images/val',
        'test': 'images/test',
        'names': {0: 'textline'},
    }
    with (dataset_root / 'dataset.yaml').open('w', encoding='utf-8') as handle:
        yaml.safe_dump(config, handle, sort_keys=False)


def summarize_dataset(dataset_root: Path) -> pd.DataFrame:
    summary = {}
    for split_name in SOURCE_SPLITS:
        image_dir = dataset_root / 'images' / split_name
        label_dir = dataset_root / 'labels' / split_name
        token_counts = Counter()
        for label_file in sorted(label_dir.glob('*.txt')):
            with label_file.open(encoding='utf-8') as handle:
                for line in handle:
                    token_counts[len(line.strip().split())] += 1
        summary[split_name] = {
            'images': len(list(image_dir.glob('*'))),
            'labels': len(list(label_dir.glob('*.txt'))),
            'polygon_rows': sum(count for tokens, count in token_counts.items() if tokens > 5),
            'bbox_rows': sum(count for tokens, count in token_counts.items() if tokens == 5),
        }
    return pd.DataFrame.from_dict(summary, orient='index')


def create_model(model_kind: str, weights: str):
    if model_kind == 'yolo':
        return YOLO(weights)
    if model_kind == 'rtdetr':
        return RTDETR(weights)
    if model_kind == 'sam':
        return SAM(weights)
    raise ValueError(f'Unsupported model family: {model_kind}')


print('COCO root:', COCO_ROOT, COCO_ROOT.exists())
print('Test PAGE dir:', TEST_PAGE_DIR, TEST_PAGE_DIR.exists())
for split_name, split_path in SOURCE_SPLITS.items():
    print(split_name, split_path, split_path.exists())
print('Run dir:', RUN_DIR)
print('Eval dir:', Path(EVAL_LOCAL))
print('Device:', DEVICE)
if ACTIVE_DATASET_ROOT is None:
    print('Active dataset: not required for SAM preview mode')
else:
    print('Active dataset:', ACTIVE_DATASET_ROOT)

In [ ]:
if ACTIVE_DATASET_ROOT is None:
    dataset_summary = None
    print('SAM uses pretrained weights only, so dataset generation is skipped.')
else:
    if FORCE_REBUILD_DATASET and ACTIVE_DATASET_ROOT.exists():
        shutil.rmtree(ACTIVE_DATASET_ROOT)

    ACTIVE_DATASET_ROOT.mkdir(parents=True, exist_ok=True)
    copy_split_images(ACTIVE_DATASET_ROOT)
    convert_coco_annotations(ACTIVE_DATASET_ROOT, use_segments=MODEL_KIND == 'yolo')
    write_dataset_yaml(ACTIVE_DATASET_ROOT)
    dataset_summary = summarize_dataset(ACTIVE_DATASET_ROOT)
    dataset_summary

In [ ]:
sample_image_path = next(iter(iter_split_images('train')))

if ACTIVE_DATASET_ROOT is None:
    image = Image.open(sample_image_path).convert('RGB')
    plt.figure(figsize=(10, 14))
    plt.imshow(image)
    plt.axis('off')
    plt.title(f'{subset_key} sample: {sample_image_path.name}')
    plt.show()
else:
    sample_label_path = ACTIVE_DATASET_ROOT / 'labels' / 'train' / f'{sample_image_path.stem}.txt'
    image = Image.open(sample_image_path).convert('RGB')
    overlay = image.copy()
    draw = ImageDraw.Draw(overlay)
    width, height = image.size

    with sample_label_path.open(encoding='utf-8') as handle:
        for row_index, line in enumerate(handle):
            values = [float(token) for token in line.strip().split()]
            if MODEL_KIND == 'yolo' and len(values) > 5:
                coords = values[1:]
                polygon = [(coords[i] * width, coords[i + 1] * height) for i in range(0, len(coords), 2)]
                draw.line(polygon + [polygon[0]], fill=(255, 0, 0), width=2)
            elif MODEL_KIND == 'rtdetr' and len(values) == 5:
                _, x_center, y_center, box_width, box_height = values
                x0 = (x_center - box_width / 2) * width
                y0 = (y_center - box_height / 2) * height
                x1 = (x_center + box_width / 2) * width
                y1 = (y_center + box_height / 2) * height
                draw.rectangle((x0, y0, x1, y1), outline=(255, 0, 0), width=2)
            if row_index >= 14:
                break

    plt.figure(figsize=(10, 14))
    plt.imshow(overlay)
    plt.axis('off')
    plt.title(f'{subset_key} sample: {sample_image_path.name}')
    plt.show()

## 5  Train or preview, then plot a random test page

In [ ]:
def run_prediction(model_kind: str, model, image_path: Path):
    predict_kwargs = {
        'source': str(image_path),
        'device': DEVICE,
        'save': False,
        'verbose': False,
    }
    if model_kind in {'yolo', 'rtdetr'}:
        predict_kwargs.update({'imgsz': IMGSZ, 'conf': PRED_CONF})
    else:
        predict_kwargs.update({'imgsz': 1024})
    return model.predict(**predict_kwargs)[0]


def iter_test_images() -> list[Path]:
    return iter_split_images('test')


def plot_random_test_prediction(model_kind: str, model, seed: int | None = None, save_path: Path | None = None) -> str | None:
    image_paths = iter_test_images()
    if not image_paths:
        print('No test images found.')
        return None

    rng = np.random.default_rng(seed)
    image_path = image_paths[int(rng.integers(0, len(image_paths)))]
    result = run_prediction(model_kind, model, image_path)
    rendered = result.plot()

    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(save_path), rendered)

    if model_kind == 'sam':
        mask_count = len(result.masks.xy) if result.masks is not None else 0
        print(f'{image_path.name}: {mask_count} masks')
    else:
        scores = result.boxes.conf.cpu().numpy() if result.boxes is not None else np.array([])
        label_name = 'text-lines' if model_kind == 'yolo' else 'detections'
        if scores.size:
            print(f'{image_path.name}: {len(scores)} {label_name} (scores min={float(scores.min()):.3f} max={float(scores.max()):.3f})')
        else:
            print(f'{image_path.name}: 0 {label_name}')

    preview = cv2.cvtColor(rendered, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 14))
    plt.imshow(preview)
    plt.axis('off')
    plt.title(f'Random test prediction: {image_path.name}')
    plt.show()
    return image_path.name


RUN_DIR.parent.mkdir(parents=True, exist_ok=True)

if MODEL_KIND == 'sam':
    train_results = None
    best_model = create_model(MODEL_KIND, MODEL_WEIGHTS)
    train_summary = {
        'model_weights': MODEL_WEIGHTS,
        'note': 'SAM is preview-only in this notebook. Training and validation are skipped.',
    }
else:
    if BEST_WEIGHTS.exists() and not FORCE_RETRAIN:
        print(f'Reusing existing weights: {BEST_WEIGHTS}')
        train_results = None
    else:
        model = create_model(MODEL_KIND, MODEL_WEIGHTS)
        train_kwargs = {
            'data': str(DATASET_YAML),
            'epochs': EPOCHS,
            'imgsz': IMGSZ,
            'batch': BATCH,
            'workers': WORKERS,
            'patience': PATIENCE,
            'device': DEVICE,
            'project': str(RUN_DIR.parent),
            'name': RUN_DIR.name,
            'exist_ok': True,
            'plots': True,
            'cache': False,
            'pretrained': True,
            'amp': True,
            'save': True,
            'verbose': True,
            'deterministic': True,
            'seed': 42,
            'cos_lr': False,
            'lr0': 0.0001,
            'optimizer': 'AdamW',
        }
        if MODEL_KIND == 'yolo':
            train_kwargs.update({
                'task': 'segment',
                'mosaic': 0.0,
                'mixup': 0.0,
                'copy_paste': 0.0,
                'fliplr': 0.0,
                'flipud': 0.0,
                'degrees': 0.0,
                'translate': 0.0,
                'scale': 0.0,
                'shear': 0.0,
                'perspective': 0.0,
                'overlap_mask': False,
            })
        else:
            train_kwargs.update({'task': 'detect'})
        train_results = model.train(**train_kwargs)

    if not BEST_WEIGHTS.exists():
        raise FileNotFoundError(
            f'Checkpoint not found at {BEST_WEIGHTS}. Rerun this cell with a lower IMGSZ if training ran out of memory.'
        )

    best_model = create_model(MODEL_KIND, str(BEST_WEIGHTS))
    val_metrics = best_model.val(
        data=str(DATASET_YAML),
        split='val',
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        plots=False,
        verbose=False,
    )
    train_summary = {
        'best_weights': str(BEST_WEIGHTS),
        'box_map50': float(val_metrics.box.map50),
        'box_map50_95': float(val_metrics.box.map),
    }
    if MODEL_KIND == 'yolo':
        train_summary.update({
            'seg_map50': float(val_metrics.seg.map50),
            'seg_map50_95': float(val_metrics.seg.map),
        })

random_image_name = plot_random_test_prediction(MODEL_KIND, best_model, seed=None)
if random_image_name is not None:
    print(f'Previewed random test image: {random_image_name}')

train_summary

In [ ]:
def load_main_textregion_polygon(gt_xml_path: Path) -> np.ndarray | None:
    if not gt_xml_path.exists():
        return None
    root = ET.parse(gt_xml_path).getroot()
    ns = {'p': PAGE_NS}
    best_poly, best_area = None, -1.0
    for region in root.iterfind('.//p:TextRegion', ns):
        coords = region.find('p:Coords', ns)
        if coords is None:
            continue
        poly = np.array(
            [list(map(int, xy.split(','))) for xy in coords.attrib['points'].split()],
            dtype=np.int32,
        )
        area = float(cv2.contourArea(poly))
        if area > best_area:
            best_poly, best_area = poly, area
    return best_poly


def simplify_polygon(points: np.ndarray, epsilon_ratio: float = 0.0025) -> list[tuple[int, int]] | None:
    pts = np.asarray(points, dtype=np.float32)
    if pts.shape[0] < 3:
        return None
    contour = pts.reshape(-1, 1, 2)
    epsilon = epsilon_ratio * cv2.arcLength(contour, True)
    polygon = cv2.approxPolyDP(contour, epsilon, True)
    if polygon.shape[0] < 3:
        polygon = contour
    return [(int(round(point[0][0])), int(round(point[0][1]))) for point in polygon]


def polygon_centroid(points: list[tuple[int, int]]) -> tuple[float, float]:
    arr = np.asarray(points, dtype=np.float32)
    return float(arr[:, 0].mean()), float(arr[:, 1].mean())


def polygon_to_baseline(points: list[tuple[int, int]]) -> list[tuple[int, int]]:
    xs = [x for x, _ in points]
    ys = [y for _, y in points]
    baseline_y = int(round(np.percentile(ys, 90)))
    return [(min(xs), baseline_y), (max(xs), baseline_y)]


def create_page_xml(image_name: str, width: int, height: int):
    root = ET.Element(
        'PcGts',
        {
            'xmlns': PAGE_NS,
            'xmlns:xsi': 'http://www.w3.org/2001/XMLSchema-instance',
            'xsi:schemaLocation': f'{PAGE_NS} {PAGE_NS}/pagecontent.xsd',
        },
    )
    metadata = ET.SubElement(root, 'Metadata')
    ET.SubElement(metadata, 'Creator').text = 'ultralytics_instance_segmentation_colab'
    page = ET.SubElement(
        root,
        'Page',
        {'imageFilename': image_name, 'imageWidth': str(width), 'imageHeight': str(height)},
    )
    region = ET.SubElement(page, 'TextRegion', {'id': 'region_text'})
    ET.SubElement(region, 'Coords', {'points': f'0,0 {width},0 {width},{height} 0,{height}'})
    return root, region


def write_prediction_pagexml(image_path: Path, image_bgr: np.ndarray, text_lines: list[dict], output_path: Path) -> None:
    height, width = image_bgr.shape[:2]
    root, region = create_page_xml(image_path.name, width, height)
    for index, line in enumerate(text_lines):
        text_line = ET.SubElement(region, 'TextLine', {'id': f'textline_{index}'})
        ET.SubElement(text_line, 'Coords', {'points': ' '.join(f'{x},{y}' for x, y in line['coords'])})
        ET.SubElement(text_line, 'Baseline', {'points': ' '.join(f'{x},{y}' for x, y in line['baseline'])})
        ET.SubElement(ET.SubElement(text_line, 'TextEquiv'), 'Unicode').text = ''

    output_path.parent.mkdir(parents=True, exist_ok=True)
    xml_str = minidom.parseString(ET.tostring(root, encoding='utf-8')).toprettyxml(indent='  ')
    output_path.write_text(xml_str, encoding='utf-8')


def parse_pagexml_polygons(xml_path: Path) -> list[list[tuple[int, int]]]:
    root = ET.parse(xml_path).getroot()
    ns = {'p': PAGE_NS}
    polygons = []
    for text_line in root.iterfind('.//p:TextLine', ns):
        coords = text_line.find('p:Coords', ns)
        if coords is None:
            continue
        polygon = [tuple(map(int, point.split(','))) for point in coords.attrib['points'].split()]
        if len(polygon) >= 3:
            polygons.append(polygon)
    return polygons


def polygon_to_mask(poly: list[tuple[int, int]], height: int, width: int) -> np.ndarray:
    mask = np.zeros((height, width), dtype=np.uint8)
    contour = np.array(poly, dtype=np.int32).reshape(-1, 1, 2)
    cv2.fillPoly(mask, [contour], 1)
    return mask.astype(bool)


def predict_text_lines(model, image_path: Path) -> list[dict]:
    region_poly = load_main_textregion_polygon(TEST_PAGE_DIR / f'{image_path.stem}.xml')
    result = run_prediction('yolo', model, image_path)
    scores = result.boxes.conf.cpu().numpy().tolist() if result.boxes is not None else []
    polygons = result.masks.xy if result.masks is not None else []
    text_lines = []

    for index, points in enumerate(polygons):
        polygon = simplify_polygon(points)
        if polygon is None or len(polygon) < 3:
            continue
        if region_poly is not None:
            cx, cy = polygon_centroid(polygon)
            if cv2.pointPolygonTest(region_poly, (cx, cy), False) < 0:
                continue
        baseline = polygon_to_baseline(polygon)
        text_lines.append({
            'coords': polygon,
            'baseline': baseline,
            'score': float(scores[index]) if index < len(scores) else 0.0,
        })

    text_lines.sort(key=lambda row: np.mean([y for _, y in row['coords']]))
    return text_lines


def export_pagexml_predictions(model) -> pd.DataFrame:
    PRED_XML_DIR.mkdir(parents=True, exist_ok=True)
    if FORCE_REEXPORT_HSCP:
        for xml_path in PRED_XML_DIR.glob('*.xml'):
            xml_path.unlink()

    rows = []
    for image_path in iter_test_images():
        image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image_bgr is None:
            continue
        text_lines = predict_text_lines(model, image_path)
        write_prediction_pagexml(image_path, image_bgr, text_lines, PRED_XML_DIR / f'{image_path.stem}.xml')
        rows.append({'image': image_path.name, 'predicted_lines': len(text_lines)})
    return pd.DataFrame(rows)


def match_score(mask_gt: np.ndarray, mask_pred: np.ndarray) -> float:
    inter = np.logical_and(mask_gt, mask_pred).sum()
    union = np.logical_or(mask_gt, mask_pred).sum()
    return inter / union if union > 0 else 0.0


def hscp_polygon_eval(gt_xml: Path, pred_xml: Path, height: int, width: int, iou_thresh: float = 0.75) -> dict:
    gt_polys = parse_pagexml_polygons(gt_xml)
    pred_polys = parse_pagexml_polygons(pred_xml)
    n_gt, n_pred = len(gt_polys), len(pred_polys)

    if n_gt == 0 or n_pred == 0:
        return {'DR': 0.0, 'RA': 0.0, 'FM': 0.0, 'M': 0, 'N_gt': n_gt, 'N_pred': n_pred}

    gt_masks = [polygon_to_mask(poly, height, width) for poly in gt_polys]
    pred_masks = [polygon_to_mask(poly, height, width) for poly in pred_polys]

    used_pred = set()
    matches = 0
    for gt_mask in gt_masks:
        best_index, best_iou = -1, 0.0
        for index, pred_mask in enumerate(pred_masks):
            if index in used_pred:
                continue
            iou = match_score(gt_mask, pred_mask)
            if iou > best_iou:
                best_index, best_iou = index, iou
        if best_iou >= iou_thresh and best_index >= 0:
            matches += 1
            used_pred.add(best_index)

    dr = matches / n_gt if n_gt else 0.0
    ra = matches / n_pred if n_pred else 0.0
    fm = (2 * dr * ra) / (dr + ra) if (dr + ra) > 0 else 0.0
    return {
        'DR': round(dr, 4),
        'RA': round(ra, 4),
        'FM': round(fm, 4),
        'M': matches,
        'N_gt': n_gt,
        'N_pred': n_pred,
    }


def score_hscp(iou_thresh: float = HSCP_IOU_THRESH) -> pd.DataFrame:
    rows = []
    for image_path in iter_test_images():
        stem = image_path.stem
        pred_xml = PRED_XML_DIR / f'{stem}.xml'
        gt_xml = TEST_PAGE_DIR / f'{stem}.xml'
        if not pred_xml.exists() or not gt_xml.exists():
            continue
        image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        height, width = image_bgr.shape[:2]
        row = hscp_polygon_eval(gt_xml, pred_xml, height, width, iou_thresh=iou_thresh)
        row['image'] = image_path.name
        rows.append(row)

    df = pd.DataFrame(rows)
    if not df.empty:
        HSCP_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(HSCP_CSV_PATH, index=False)
        print(f"Mean DR: {df['DR'].mean():.4f}  RA: {df['RA'].mean():.4f}  FM: {df['FM'].mean():.4f}")
    return df


if MODEL_KIND not in HSCP_MODEL_FAMILIES:
    prediction_summary = None
    hscp_results = None
    print('HSCP evaluation is only available for YOLO segmentation in this notebook, because it needs polygon predictions.')
else:
    if not BEST_WEIGHTS.exists():
        raise FileNotFoundError(f'Missing trained weights: {BEST_WEIGHTS}')
    best_model = create_model('yolo', str(BEST_WEIGHTS))
    prediction_summary = export_pagexml_predictions(best_model)
    hscp_results = score_hscp()
    hscp_metrics = pd.DataFrame(
        [
            {
                'subset': subset_key,
                'n_test_images': int(len(iter_test_images())),
                'n_prediction_files': int(len(list(PRED_XML_DIR.glob('*.xml')))),
                'dr': float(hscp_results['DR'].mean()) if not hscp_results.empty else None,
                'ra': float(hscp_results['RA'].mean()) if not hscp_results.empty else None,
                'fm': float(hscp_results['FM'].mean()) if not hscp_results.empty else None,
            }
        ]
    )
    hscp_metrics, prediction_summary.head()

## 6  HSCP evaluation (YOLO segmentation only)

## 7  Sync outputs back to Drive

In [ ]:
out_drive = Path(OUTPUT_DRIVE)
out_drive.parent.mkdir(parents=True, exist_ok=True)

if RUN_DIR.exists():
    if out_drive.exists():
        shutil.rmtree(out_drive)
    shutil.copytree(RUN_DIR, out_drive)
    print('synced run to', out_drive)
else:
    print('no trainable run directory to sync')

if Path(EVAL_LOCAL).exists():
    eval_drive = Path(EVAL_DRIVE)
    eval_drive.parent.mkdir(parents=True, exist_ok=True)
    if eval_drive.exists():
        shutil.rmtree(eval_drive)
    shutil.copytree(Path(EVAL_LOCAL), eval_drive)
    print('synced evaluation to', eval_drive)

## 8  Save another random test preview

In [ ]:
if MODEL_KIND == 'sam':
    best_model = create_model(MODEL_KIND, MODEL_WEIGHTS)
else:
    if not BEST_WEIGHTS.exists():
        raise FileNotFoundError(f'Missing trained weights: {BEST_WEIGHTS}')
    best_model = create_model(MODEL_KIND, str(BEST_WEIGHTS))

preview_local = Path(OUTPUT_LOCAL) / 'pred_random_test.jpg'
preview_drive = Path(OUTPUT_DRIVE) / preview_local.name

random_image_name = plot_random_test_prediction(MODEL_KIND, best_model, seed=None, save_path=preview_local)
if preview_local.exists():
    preview_drive.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(preview_local, preview_drive)
    print(f'saved {random_image_name} to {preview_local}')
    print(f'saved {random_image_name} to {preview_drive}')